In [1]:
# ==============================================================================
# CELDA 1: Importaciones y Configuración Inicial
# ==============================================================================
import numpy as np
import cupy as cp
import matplotlib.pyplot as plt
from scipy.special import lambertw
from scipy.stats import linregress
import time
import gc

# Configuración de estilo para las gráficas
plt.style.use('dark_background')
plt.rcParams['figure.dpi'] = 120

print("✅ Librerías importadas correctamente.")
try:
    gpu_name = cp.cuda.runtime.getDeviceProperties(0)['name'].decode('utf-8')
    print(f"✅ GPU detectada: {gpu_name}")
except Exception as e:
    print("❌ No se ha detectado GPU. Ve a 'Entorno de ejecución' > 'Cambiar tipo de entorno de ejecución' y selecciona 'T4 GPU'.")


✅ Librerías importadas correctamente.
✅ GPU detectada: Tesla T4


In [2]:
# ==============================================================================
# CELDA 2: Funciones Core del Hamiltoniano Riemann-GUE y SFF
# ==============================================================================

def build_rgue_hamiltonian_gpu(N, epsilon=np.pi*np.sqrt(2), nu=0.75):
    """Construye el Hamiltoniano RGUE optimizado para GPU usando CuPy."""
    # 1. Diagonal: Inversión exacta de Lambert W
    k = np.arange(2, N + 2)
    k_eff = k - 7/8
    diag_E = (2 * np.pi * k_eff) / np.real(lambertw(k_eff / np.e))
    H = cp.diag(cp.array(diag_E, dtype=cp.float32)).astype(cp.complex64)

    # 2. Índices fuera de la diagonal (triángulo superior)
    row, col = cp.triu_indices(N, k=1)
    d = col - row

    # 3. Máscara Topológica Z/6Z (canales 1 y 5)
    mask = cp.isin(d % 6, cp.array([1, 5]))

    row_m = row[mask]
    col_m = col[mask]
    d_m = d[mask]

    # 4. Ruido GUE y Atenuación PRBM (Kato-Rellich)
    num_elements = len(d_m)
    G = (cp.random.randn(num_elements, dtype=cp.float32) +
         1j * cp.random.randn(num_elements, dtype=cp.float32)) / cp.sqrt(2)

    decay = cp.power(d_m, -nu, dtype=cp.float32)
    vals = epsilon * decay * G

    # 5. Rellenar matriz de forma hermítica
    H[row_m, col_m] = vals
    H[col_m, row_m] = cp.conj(vals)

    return H

def unfold_spectrum(evals):
    """Despliegue del espectro usando la fórmula teórica de Riemann-von Mangoldt."""
    # Evitamos logaritmos de números negativos o cero
    evals_safe = np.where(evals > 2*np.pi*np.e, evals, 2*np.pi*np.e + 1e-5)
    w_n = (evals_safe / (2 * np.pi)) * np.log(evals_safe / (2 * np.pi * np.e))
    return w_n

def calculate_sff(w_n, t_values):
    """Calcula el Spectral Form Factor para un espectro desplegado dado."""
    N = len(w_n)
    # Broadcasting en CPU para evitar saturar la RAM de la GPU
    phases = np.exp(-1j * w_n[:, None] * t_values[None, :])
    K_t = np.abs(np.sum(phases, axis=0))**2 / N
    return K_t

print("✅ Funciones Core compiladas y listas para ejecución.")

✅ Funciones Core compiladas y listas para ejecución.


In [3]:
# ==============================================================================
# CELDA 3: Bucle Principal de Ensamblaje (100 Realizaciones x N=15000)
# ==============================================================================

# Parámetros del experimento
N = 15000           # Tamaño MASIVO de la matriz
M = 100             # Número de realizaciones a promediar
t_values = np.logspace(-2, 1.5, 300) # Vector de tiempos

# Acumuladores
K_t_total = np.zeros_like(t_values, dtype=np.float64)
ipr_total = 0.0

print("="*60)
print(f"🚀 Iniciando Promedio de Ensamble Termodinámico")
print(f"   Dimensión: N = {N} | Realizaciones: M = {M}")
print("   (NOTA: Este proceso puede tardar ~2 horas en una T4)")
print("="*60)

t_start_ensamble = time.time()

for m in range(M):
    t_mat_start = time.time()

    # 1. Generar matriz en GPU
    H_gpu = build_rgue_hamiltonian_gpu(N)

    # 2. Diagonalización exacta en GPU
    evals_gpu, evecs_gpu = cp.linalg.eigh(H_gpu)

    # 3. Transferir a CPU
    evals = evals_gpu.get()
    evecs = evecs_gpu.get()

    # 4. Calcular IPR en el "bulk" (centro del espectro)
    bulk_start, bulk_end = int(0.25 * N), int(0.75 * N)
    bulk_evecs = evecs[:, bulk_start:bulk_end]
    ipr_bulk = np.mean(np.sum(np.abs(bulk_evecs)**4, axis=0))
    ipr_total += ipr_bulk

    # 5. Desplegar espectro y calcular SFF
    w_n = unfold_spectrum(evals)
    K_t = calculate_sff(w_n, t_values)
    K_t_total += K_t

    # 6. Limpieza profunda de memoria (CRÍTICO PARA N=15000)
    del H_gpu, evals_gpu, evecs_gpu, evals, evecs, bulk_evecs, w_n, K_t
    cp.get_default_memory_pool().free_all_blocks()
    gc.collect()

    t_mat_end = time.time()

    # Progreso
    print(f" -> Realización {m + 1:03d}/{M} completada en {t_mat_end - t_mat_start:.2f}s | IPR Local: {ipr_bulk:.6f}")

# Cálculos finales promediados
K_t_avg = K_t_total / M
ipr_avg = ipr_total / M
D2_fractal = -np.log(ipr_avg) / np.log(N)

total_time = (time.time() - t_start_ensamble) / 60

print("\n" + "="*60)
print(f"✅ ENSAMBLE COMPLETADO (Tiempo total: {total_time:.2f} minutos)")
print("="*60)

🚀 Iniciando Promedio de Ensamble Termodinámico
   Dimensión: N = 15000 | Realizaciones: M = 100
   (NOTA: Este proceso puede tardar ~2 horas en una T4)
 -> Realización 001/100 completada en 42.45s | IPR Local: 0.096045
 -> Realización 002/100 completada en 39.47s | IPR Local: 0.096525
 -> Realización 003/100 completada en 39.23s | IPR Local: 0.097433
 -> Realización 004/100 completada en 39.93s | IPR Local: 0.095747
 -> Realización 005/100 completada en 40.47s | IPR Local: 0.096446
 -> Realización 006/100 completada en 41.44s | IPR Local: 0.096903
 -> Realización 007/100 completada en 40.51s | IPR Local: 0.097084
 -> Realización 008/100 completada en 40.41s | IPR Local: 0.095915
 -> Realización 009/100 completada en 41.07s | IPR Local: 0.096213
 -> Realización 010/100 completada en 40.29s | IPR Local: 0.097312
 -> Realización 011/100 completada en 40.42s | IPR Local: 0.095626
 -> Realización 012/100 completada en 40.49s | IPR Local: 0.096488
 -> Realización 013/100 completada en 40.56s

In [4]:
# ==============================================================================
# CELDA 4: Análisis Cuantitativo, Resultados Destacados y Gráficas
# ==============================================================================

# 1. Identificar la región de la rampa para el ajuste (aprox entre t=0.2 y t=2.0)
ramp_mask = (t_values > 0.2) & (t_values < 2.0)
t_ramp = t_values[ramp_mask]
K_ramp = K_t_avg[ramp_mask]

# 2. Ajuste lineal en escala log-log para extraer la pendiente (gamma)
log_t = np.log(t_ramp)
log_K = np.log(K_ramp)
slope, intercept, r_value, _, _ = linregress(log_t, log_K)

# 3. Análisis de la Meseta
mask_plateau = t_values > (2 * np.pi * 1.1)
plateau_mean = np.mean(K_t_avg[mask_plateau])

# --- IMPRESIÓN DE RESULTADOS CLAVE AL USUARIO ---
print("\n" + "★"*60)
print("      RESULTADOS FÍSICOS DEL ENSAMBLE (N=15000, M=100)")
print("★"*60)
print(f" 1. Dimensión Fractal Generalizada:  D2 = {D2_fractal:.4f}")
print(f"    (D2 < 1 demuestra confinamiento multifractal en el Bulk)")
print("")
print(f" 2. Exponente de Difusión Anómala:   γ  = {slope:.4f}")
print(f"    (Pendiente de la rampa muy inferior a 1.0 -> Fase NEE)")
print("")
print(f" 3. Saturación de la Meseta (t>2π):  K  = {plateau_mean:.4f}")
print(f"    (Confirma que el espectro es estrictamente discreto/real)")
print("★"*60 + "\n")

# --- GRÁFICA ---
plt.figure(figsize=(10, 6))

# SFF Promediado
plt.plot(t_values, K_t_avg, label=rf'Averaged SFF $\langle K(t) \rangle$ ({M} Matrices)', color='cyan', lw=2)

# Ajuste de la rampa anómala (Uso de 'rf' soluciona el SyntaxWarning)
fit_line = np.exp(intercept) * t_ramp**slope
plt.plot(t_ramp, fit_line, color='magenta', lw=2.5, linestyle='--',
         label=rf'Fractional Ramp Fit: $\gamma \approx {slope:.3f}$')

# Líneas teóricas de referencia (Uso de 'r' soluciona el SyntaxWarning)
plt.axvline(x=2*np.pi, color='red', linestyle=':', label=r'Heisenberg Time $t_H = 2\pi$')
plt.axhline(y=1.0, color='gray', linestyle='-.', alpha=0.5, label='Ergodic Plateau $K(t)=1$')
plt.plot(t_ramp, 0.5 * t_ramp**1.0, color='white', lw=1.5, linestyle=':', alpha=0.7,
         label=r'Standard GUE ($\gamma = 1.0$)')

plt.xscale('log')
plt.yscale('log')
plt.xlabel('Time $t$ (log scale)', fontsize=12)
plt.ylabel(r'Spectral Form Factor $\langle K(t) \rangle$', fontsize=12)
plt.title(rf'Ensemble Averaged SFF ($N={N}, M={M}$)', fontsize=14, fontweight='bold')
plt.legend(loc='lower right')
plt.grid(True, which="both", ls="--", alpha=0.2)
plt.tight_layout()

# Guardar figura en alta calidad
filename = 'PRL_Figure_SFF_Ensemble_15k.png'
plt.savefig(filename, dpi=300)
plt.show()

print(f"✅ Gráfica guardada como '{filename}'. Lista para incluir en el paper.")


★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★
      RESULTADOS FÍSICOS DEL ENSAMBLE (N=15000, M=100)
★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★
 1. Dimensión Fractal Generalizada:  D2 = 0.2433
    (D2 < 1 demuestra confinamiento multifractal en el Bulk)

 2. Exponente de Difusión Anómala:   γ  = 0.3203
    (Pendiente de la rampa muy inferior a 1.0 -> Fase NEE)

 3. Saturación de la Meseta (t>2π):  K  = 0.9989
    (Confirma que el espectro es estrictamente discreto/real)
★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★



✅ Gráfica guardada como 'PRL_Figure_SFF_Ensemble_15k.png'. Lista para incluir en el paper.
